## 1. Instalación de dependencias

In [ ]:
!pip install -q -U \
    "transformers>=4.47.0" \
    "Pillow>=10.4.0" \
    torchvision \
    accelerate \
    sentencepiece \
    fastapi \
    uvicorn \
    python-multipart \
    pyngrok \
    groq \
    huggingface_hub \
    nest_asyncio
print("All packages installed.")

## 2. Configuración de credenciales (variables de entorno)

In [ ]:
import os
from google.colab import userdata

HF_TOKEN     = userdata.get("HF_TOKEN")
GROQ_API_KEY = userdata.get("GROQ_API_KEY")
NGROK_TOKEN  = userdata.get("NGROK_TOKEN")

if not HF_TOKEN:     raise ValueError("Missing HF_TOKEN — add it in Colab Secrets (key icon)")
if not GROQ_API_KEY: raise ValueError("Missing GROQ_API_KEY — add it in Colab Secrets (key icon)")
if not NGROK_TOKEN:  raise ValueError("Missing NGROK_TOKEN — add it in Colab Secrets (key icon)")

os.environ["HF_TOKEN"]          = HF_TOKEN
os.environ["GROQ_API_KEY"]      = GROQ_API_KEY
os.environ["NGROK_AUTH_TOKEN"]  = NGROK_TOKEN

print("✓ Secrets loaded correctly.")

## Login en Hugging Face

In [ ]:
from huggingface_hub import login, whoami

# Login explícito con token para evitar 401 en repos gated
login(token=os.environ["HF_TOKEN"], add_to_git_credential=False)
print("HF user:", whoami()["name"])

## Creación del backend (server.py)

In [ ]:
%%writefile server.py
import os
import io
import tempfile
import torch
from transformers import AutoProcessor, PaliGemmaForConditionalGeneration
from fastapi import FastAPI, File, UploadFile, Form
from fastapi.middleware.cors import CORSMiddleware
from PIL import Image
from groq import Groq

# ── Credenciales ───────────────────────────────────────────────────────────
GROQ_API_KEY = os.environ["GROQ_API_KEY"]
HF_TOKEN     = os.environ["HF_TOKEN"]
groq_client  = Groq(api_key=GROQ_API_KEY)

# ── PaliGemma 2 — 3B mix 448 ───────────────────────────────────────────────
MODEL_ID = "google/paligemma2-3b-mix-448"
DEVICE   = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE    = torch.bfloat16 if DEVICE == "cuda" else torch.float32

print(f"Device: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU : {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

print(f"\nLoading {MODEL_ID} ...")
processor = AutoProcessor.from_pretrained(MODEL_ID, token=HF_TOKEN)
model = PaliGemmaForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    device_map="auto",
    token=HF_TOKEN,
).eval()
print("PaliGemma 2 loaded ✓")
if DEVICE == "cuda":
    used  = torch.cuda.memory_allocated() / 1024**3
    total = torch.cuda.get_device_properties(0).total_memory / 1024**3
    print(f"GPU memory used: {used:.1f} / {total:.1f} GB")

# ── FastAPI ────────────────────────────────────────────────────────────────
app = FastAPI(title="VisionLeader UFV — Colab Backend")
app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_methods=["*"],
    allow_headers=["*"],
)

# Prompt de dominio para que Whisper reconozca vocabulario del campus UFV
WHISPER_DOMAIN_PROMPT = (
    "UFV campus navigation assistant. "
    "Building G, Building CD, Building H. "
    "Ground floor, first floor, second floor, basement. "
    "Classroom 0.1, 0.3, 1.1, 1.4, 2.1. "
    "Elevator, stairs, cafeteria, gym, restroom, AI laboratory, "
    "secretary office, mentoring room, main entrance, directory sign. "
    "Yes, no, done, repeat, cancel, I am lost, I am in a hurry, "
    "I want to go to, take me to, navigate to, I am in."
)


@app.get("/")
async def root():
    return {
        "status": "running",
        "model": MODEL_ID,
        "device": DEVICE,
        "endpoints": ["/analyze", "/transcribe"],
    }


@app.post("/analyze")
async def analyze_image(
    file: UploadFile = File(...),
    prompt: str = Form(None),
    max_tokens: str = Form(None),
):
    """
    Analiza una imagen con PaliGemma 2.
    Recibe: file (imagen), prompt (texto), max_tokens (opcional).
    Devuelve: {"success": bool, "response": str, "error": str|null}
    """
    try:
        image_bytes = await file.read()
        image = Image.open(io.BytesIO(image_bytes)).convert("RGB")
        max_new = int(max_tokens) if (max_tokens and max_tokens.isdigit()) else 256

        prompt_to_use = prompt or (
            "Answer in this exact format: Building: [CODE], Floor: [NUMBER]\n"
            "From this campus directory sign, identify the building code "
            "(G, CD, or H) and the floor number (-1, 0, 1, or 2)."
        )
        # PaliGemma 2 requires <image> prefix
        if not prompt_to_use.strip().startswith("<image>"):
            prompt_to_use = "<image>\n" + prompt_to_use

        print(f"\n[/analyze] prompt={prompt_to_use[:150]!r}")

        inputs = processor(
            text=prompt_to_use,
            images=image,
            padding="longest",
            return_tensors="pt",
        ).to(model.device)
        inputs = inputs.to(dtype=model.dtype)

        with torch.no_grad():
            outputs = model.generate(
                **inputs,
                max_new_tokens=max_new,
                do_sample=False,
                num_beams=1,
                pad_token_id=(
                    processor.tokenizer.pad_token_id
                    or processor.tokenizer.eos_token_id
                ),
                eos_token_id=processor.tokenizer.eos_token_id,
            )

        input_len = inputs["input_ids"].shape[-1]
        response = processor.decode(
            outputs[0][input_len:], skip_special_tokens=True
        ).strip()

        print(f"[/analyze] response={response!r}")
        return {"success": True, "response": response, "error": None}

    except Exception as e:
        print(f"[/analyze] ERROR: {e}")
        return {"success": False, "response": "", "error": str(e)}


@app.post("/transcribe")
async def transcribe_audio(file: UploadFile = File(...)):
    """
    Transcribe audio con Groq Whisper large-v3.
    Recibe: file (audio WAV).
    Devuelve: {"success": bool, "text": str, "error": str|null}
    """
    try:
        audio_data = await file.read()
        fname = file.filename or "audio.wav"
        print(f"\n[/transcribe] file={fname}  size={len(audio_data)} bytes")

        with tempfile.NamedTemporaryFile(delete=False, suffix=".wav") as tmp:
            tmp.write(audio_data)
            tmp_path = tmp.name

        try:
            with open(tmp_path, "rb") as audio_file:
                result = groq_client.audio.transcriptions.create(
                    file=audio_file,
                    model="whisper-large-v3",   # mejor modelo disponible
                    response_format="text",
                    language="en",
                    temperature=0.0,            # determinista, sin alucinaciones
                    prompt=WHISPER_DOMAIN_PROMPT,
                )
            text = result.strip() if isinstance(result, str) else ""
        finally:
            os.unlink(tmp_path)

        print(f"[/transcribe] text={text!r}")
        return {"success": True, "text": text, "error": None}

    except Exception as e:
        print(f"[/transcribe] ERROR: {e}")
        return {"success": False, "text": "", "error": str(e)}


## Configuración de ngrok

In [ ]:
import time
from pyngrok import ngrok, conf

PORT = 8000

# Configurar ngrok
conf.get_default().auth_token = os.environ["NGROK_AUTH_TOKEN"]

# Cerrar túneles previos
try:
    for t in ngrok.get_tunnels():
        ngrok.disconnect(t.public_url)
except Exception:
    pass
ngrok.kill()
time.sleep(2)

# Abrir túnel
tunnel = ngrok.connect(PORT)
PUBLIC_URL = tunnel.public_url

print("=" * 60)
print(f"  URL pública: {PUBLIC_URL}")
print(f"\n  .env de Windows:")
print(f"  COLAB_API_URL={PUBLIC_URL}")
print("=" * 60)
print("\n  Ejecuta ahora la celda siguiente para arrancar el servidor.")

## Ejecución de Uvicorn

In [ ]:
# Esta celda se queda girando mientras el servidor está activo.
# Para pararlo: Runtime → Interrupt execution (o el botón ■)
import nest_asyncio
import uvicorn

nest_asyncio.apply()  # permite uvicorn dentro del event loop de Colab

print("Arrancando servidor... (tarda 3-5 min cargando PaliGemma la primera vez)")
print("Cuando veas 'Application startup complete.' ya está listo.\n")

config = uvicorn.Config("server:app", host="0.0.0.0", port=8000, log_level="info")
server = uvicorn.Server(config)
await server.serve()

In [ ]:
# Celda de verificación — ejecuta SOLO después de ver "Application startup complete"
import requests
try:
    r = requests.get(f"{PUBLIC_URL}/", timeout=5)
    print("✓ Servidor listo:", r.json())
except Exception as e:
    print(f"✗ Aún no está listo — espera y vuelve a ejecutar: {e}")